# Project 02: SQL for Data Engineering - Interactive Notebook

Welcome to the interactive companion notebook for **SQL for Data Engineering** in the Data Engineering Diaries series by Sathiya Murthy.

This notebook demonstrates setup, schema execution, and querying against an in-memory SQLite analytical warehouse.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

# Set display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 1. Initialize Warehouse Database
We create an in-memory SQLite database and populate it with schema DDL and seed datasets.

In [ ]:
conn = sqlite3.connect(':memory:')

schema_path = Path('../schemas/schema_ddl.sql')
seed_path = Path('../datasets/seed_data.sql')

if schema_path.exists() and seed_path.exists():
    conn.executescript(schema_path.read_text(encoding='utf-8'))
    conn.executescript(seed_path.read_text(encoding='utf-8'))
    print("✅ Database schema and seed data loaded successfully!")
else:
    print("⚠️ Schema/seed files not found at expected relative paths.")

## 2. Inspect Database Tables

In [ ]:
tables_df = pd.read_sql_query("SELECT name, type FROM sqlite_master WHERE type='table' ORDER BY name;", conn)
tables_df

## 3. Query Fact & Dimension Tables (Star Schema)
Run an analytical query calculating total revenue per customer segment.

In [ ]:
query = """
SELECT 
    c.customer_segment,
    COUNT(DISTINCT c.customer_id) AS total_customers,
    COUNT(s.sales_id) AS total_orders,
    COALESCE(SUM(s.total_amount), 0) AS total_revenue
FROM dim_customer c
LEFT JOIN fact_sales s ON c.customer_id = s.customer_id
GROUP BY c.customer_segment
ORDER BY total_revenue DESC;
"""
df_result = pd.read_sql_query(query, conn)
df_result

## 4. Window Functions: Top Customers per Segment

In [ ]:
query_wf = """
WITH customer_revenue AS (
    SELECT 
        c.customer_segment,
        c.first_name || ' ' || c.last_name AS customer_name,
        SUM(s.total_amount) AS revenue,
        DENSE_RANK() OVER (PARTITION BY c.customer_segment ORDER BY SUM(s.total_amount) DESC) AS rnk
    FROM dim_customer c
    JOIN fact_sales s ON c.customer_id = s.customer_id
    GROUP BY c.customer_segment, customer_name
)
SELECT customer_segment, customer_name, revenue, rnk
FROM customer_revenue
WHERE rnk <= 3;
"""
df_top = pd.read_sql_query(query_wf, conn)
df_top